In [1]:
from manim import *
from scipy.integrate import odeint
import numpy as np

# ---------------------------------------------------------
# GLOBAL CONFIGURATION: EXPAND THE CANVAS
# ---------------------------------------------------------
config.frame_height = 10.0
config.frame_width = 18.0
config.pixel_height = 1080
config.pixel_width = 1920

class DiffEqExplained(Scene):
    def construct(self):
        # ---------------------------------------------------------
        # 1. LAYOUT & ZONES
        # ---------------------------------------------------------
        
        # --- HEADER ZONE (Top) ---
        # MOVED UP: buff=0.5 (was 0.8) to pull it away from the graph
        eq_a = MathTex("a \\frac{d^3y}{dx^3}", color=RED)
        eq_b = MathTex("+", "b \\frac{d^2y}{dx^2}", color=ORANGE)
        eq_c = MathTex("+", "c \\frac{dy}{dx}", color=GREEN)
        eq_k = MathTex("=", "k", color=YELLOW)
        
        equation = VGroup(eq_a, eq_b, eq_c, eq_k).arrange(RIGHT, buff=0.2)
        # SCALED DOWN SLIGHTLY: 1.1 instead of 1.2 to reduce clutter
        equation.scale(1.1).to_edge(UP, buff=0.5).to_edge(LEFT, buff=1.0)

        # --- GRAPH ZONE (Top Left) ---
        # MOVED DOWN: buff=2.8 (was 2.0) to create a big gap between Equation and Graph
        # RESIZED: y_length=4.5 (was 5.0) to avoid hitting the bottom text box
        axes = Axes(
            x_range=[0, 10, 1],
            y_range=[-4, 12, 2],
            x_length=7.5, 
            y_length=4.5, 
            axis_config={"include_tip": False, "font_size": 22, "color": GREY},
        ).to_edge(LEFT, buff=1.0).to_edge(UP, buff=2.8)
        
        labels = axes.get_axis_labels(x_label="Time (s)", y_label="Force (N)")
        
        # --- ROAD ZONE (Far Right) ---
        road_line = Line(UP*5, DOWN*5, color=WHITE, stroke_width=5).to_edge(RIGHT, buff=3.5)
        road_dashed = DashedLine(UP*5, DOWN*5, color=GREY).to_edge(RIGHT, buff=3.5)
        
        # --- CAPTION ZONE (Bottom Left) ---
        # Pushed slightly further down (buff=0.3)
        caption_bg = Rectangle(height=2.5, width=10, color=BLACK, fill_opacity=0.8)
        caption_bg.to_edge(DOWN, buff=0.3).to_edge(LEFT, buff=1.0)
        self.add(caption_bg)

        # ---------------------------------------------------------
        # 2. HELPER FUNCTIONS
        # ---------------------------------------------------------
        def update_caption(title, body, color=WHITE):
            group = VGroup(
                Text(title, font_size=32, color=color, weight=BOLD),
                Text(body, font_size=22, color=WHITE, line_spacing=1.3)
            ).arrange(DOWN, aligned_edge=LEFT)
            group.move_to(caption_bg.get_center())
            return group

        # ---------------------------------------------------------
        # 3. PHYSICS ENGINE
        # ---------------------------------------------------------
        k_val = 8.0   
        a_val = 1.0   
        b_val = 0.6   
        c_val = 2.5   

        def model(state, t):
            y, v, acc = state
            jerk = (k_val - c_val*v - b_val*acc) / a_val
            return [v, acc, jerk]

        t_span = np.linspace(0, 10, 300)
        sol = odeint(model, [0, 0, 0], t_span)
        
        y_data = sol[:, 0]
        v_data = sol[:, 1]
        acc_data = sol[:, 2]
        
        f_fric = c_val * v_data     
        f_inert = b_val * acc_data  
        f_jerk = k_val - f_fric - f_inert 

        # ---------------------------------------------------------
        # 4. BUS VISUALIZER
        # ---------------------------------------------------------
        def get_bus(y, v, drag):
            sy = -4.0 + (y * 0.25)
            if sy > 4.0: sy = 4.0
            
            center = road_line.get_center()
            center[1] = sy
            
            body = RoundedRectangle(corner_radius=0.2, height=1.6, width=0.9, color=ORANGE, fill_opacity=1).move_to(center)
            win = Rectangle(height=0.4, width=0.7, color=BLACK, fill_opacity=0.5).move_to(center + UP*0.4)
            w1 = Circle(radius=0.15, color=GREY, fill_opacity=1).move_to(center + DOWN*0.8 + LEFT*0.45)
            w2 = Circle(radius=0.15, color=GREY, fill_opacity=1).move_to(center + DOWN*0.8 + RIGHT*0.45)
            
            arr_eng = Arrow(center, center + UP*1.8, color=YELLOW, buff=0, stroke_width=8, max_tip_length_to_length_ratio=0.2)
            arr_drag = Arrow(center, center + DOWN*(drag*0.2 + 0.1), color=GREEN, buff=0, stroke_width=8, max_tip_length_to_length_ratio=0.2)
            
            return VGroup(w1, w2, body, win, arr_eng, arr_drag)

        # ---------------------------------------------------------
        # 5. INTRO SEQUENCE
        # ---------------------------------------------------------
        
        # Term A
        self.play(Write(equation[0])) 
        cap = update_caption("THE JOLT (a)", "Term: d³y/dx³\nThis is the 'Kick'. It resists the initial change in acceleration.\nDominates only at the very start.", RED)
        self.play(FadeIn(cap)); self.wait(3)
        
        # Term B
        self.play(Write(equation[1]), FadeOut(cap)) 
        cap = update_caption("THE INERTIA (b)", "Term: Mass * Acceleration\nOnce moving, the engine must fight the weight of the bus.\nDominates during the speed-up phase.", ORANGE)
        self.play(FadeIn(cap)); self.wait(3)
        
        # Term C
        self.play(Write(equation[2]), FadeOut(cap))
        cap = update_caption("THE DRAG (c)", "Term: Friction * Velocity\nAir resistance. The faster the bus goes,\nthe harder the air pushes back.", GREEN)
        self.play(FadeIn(cap)); self.wait(3)
        
        # Term K
        self.play(Write(equation[3]), FadeOut(cap))
        cap = update_caption("THE ENGINE (k)", "Input Power.\nA constant force (gas pedal floored).\nThis fixed budget is shared by Jolt, Inertia, and Drag.", YELLOW)
        self.play(FadeIn(cap)); self.wait(3)
        self.play(FadeOut(cap))
        
        # Show Graph and Road
        self.play(Create(axes), Write(labels), Create(road_line), Create(road_dashed))

        # ---------------------------------------------------------
        # 6. ANIMATION LOOP
        # ---------------------------------------------------------
        tracker = ValueTracker(0)
        
        # Curves
        curve_j = axes.plot_line_graph(t_span, f_jerk, line_color=RED, add_vertex_dots=False)
        curve_m = axes.plot_line_graph(t_span, f_inert, line_color=ORANGE, add_vertex_dots=False)
        curve_f = axes.plot_line_graph(t_span, f_fric, line_color=GREEN, add_vertex_dots=False)
        line_k = axes.plot(lambda x: k_val, color=YELLOW)
        
        # Dots
        dot_j = always_redraw(lambda: Dot(axes.c2p(tracker.get_value(), np.interp(tracker.get_value(), t_span, f_jerk)), color=RED))
        dot_m = always_redraw(lambda: Dot(axes.c2p(tracker.get_value(), np.interp(tracker.get_value(), t_span, f_inert)), color=ORANGE))
        dot_f = always_redraw(lambda: Dot(axes.c2p(tracker.get_value(), np.interp(tracker.get_value(), t_span, f_fric)), color=GREEN))
        
        # Scanner
        scanner = always_redraw(lambda: Line(axes.c2p(tracker.get_value(), -4), axes.c2p(tracker.get_value(), 12), color=WHITE, stroke_width=1, stroke_opacity=0.5))
        
        # Bus
        bus = always_redraw(lambda: get_bus(
            np.interp(tracker.get_value(), t_span, y_data), 
            np.interp(tracker.get_value(), t_span, v_data), 
            np.interp(tracker.get_value(), t_span, f_fric)
        ))

        self.add(curve_j, curve_m, curve_f, line_k, dot_j, dot_m, dot_f, scanner, bus)

        # ---------------------------------------------------------
        # 7. STOPS
        # ---------------------------------------------------------

        # STOP 1
        cap = update_caption("T=0: IGNITION", "The engine provides Power (Yellow Line).\nBus is stopped. 100% of force goes into Jolt.\nDrag and Inertia are zero.", YELLOW)
        self.play(FadeIn(cap)); self.wait(2); self.play(FadeOut(cap))

        # STOP 2
        self.play(tracker.animate.set_value(0.5), run_time=2, rate_func=linear)
        cap = update_caption("T=0.5s: THE SNAP (a)", "Max Jolt (Red Peak).\nThis is the 'Whiplash' moment.\nThe bus barely moves, but the internal force is violent.", RED)
        lbl = Text("Peak Jolt", font_size=24, color=RED).next_to(dot_j, RIGHT, buff=0.2)
        bus_lbl = Text("FEELING: SNAP!", font_size=30, color=RED).next_to(bus, LEFT, buff=1.0)
        self.play(FadeIn(cap), Write(lbl), Write(bus_lbl))
        self.wait(4)
        self.play(FadeOut(cap), FadeOut(lbl), FadeOut(bus_lbl))

        # STOP 3
        peak_t = t_span[np.argmax(f_inert)]
        self.play(tracker.animate.set_value(peak_t), run_time=2.5, rate_func=linear)
        cap = update_caption("T=1.8s: HEAVY LIFT (b)", "Max Acceleration (Orange Peak).\nThe Jolt is gone. Now the engine battles Mass.\nPassengers are pressed firmly into seats.", ORANGE)
        lbl = Text("Max Push", font_size=24, color=ORANGE).next_to(dot_m, UP, buff=0.2)
        bus_lbl = Text("FEELING: HEAVY", font_size=30, color=ORANGE).next_to(bus, LEFT, buff=1.0)
        self.play(FadeIn(cap), Write(lbl), Write(bus_lbl))
        self.wait(4)
        self.play(FadeOut(cap), FadeOut(lbl), FadeOut(bus_lbl))

        # STOP 4
        self.play(tracker.animate.set_value(4.0), run_time=3, rate_func=linear)
        cap = update_caption("T=4.0s: DRAG RISING (c)", "Speed is high, so Air Resistance (Green) spikes.\nFriction is stealing energy from Acceleration.\nThe bus struggles to go faster.", GREEN)
        lbl = Text("Drag > Inertia", font_size=24, color=GREEN).next_to(dot_f, LEFT, buff=0.2)
        bus_lbl = Text("NOISE: WIND", font_size=30, color=GREEN).next_to(bus, LEFT, buff=1.0)
        self.play(FadeIn(cap), Write(lbl), Write(bus_lbl))
        self.wait(4)
        self.play(FadeOut(cap), FadeOut(lbl), FadeOut(bus_lbl))

        # STOP 5
        self.play(tracker.animate.set_value(10.0), run_time=4, rate_func=linear)
        cap = update_caption("T=10s: CRUISING", "Terminal Velocity.\nEngine Force (Yellow) = Drag Force (Green).\nAcceleration is zero. The ride is smooth.", YELLOW)
        lbl = Text("Balanced", font_size=24, color=YELLOW).next_to(dot_f, UP, buff=0.2)
        bus_lbl = Text("FEELING: SMOOTH", font_size=30, color=YELLOW).next_to(bus, LEFT, buff=1.0)
        self.play(FadeIn(cap), Write(lbl), Write(bus_lbl))
        self.wait(5)


%manim -qk -v warning DiffEqExplained

Manim Community v0.19.0